# Modelagem — XGBoost Multiclasse
## Classificação da Progressão da Doença de Parkinson por Telemonitoramento de Voz

**Disciplina:** Ciência de Dados — Centro Universitário Dom Helder  
**Orientador:** Prof. Dr. Marcos W. Rodrigues  

Este notebook implementa as etapas de pré-processamento, treinamento, otimização de hiperparâmetros via `GridSearchCV` e avaliação completa do modelo XGBoost para classificação multiclasse.

---

## 1. Importações e Configuração

In [ ]:
import sys, os, json, warnings
sys.path.insert(0, '../src/data')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier, plot_importance
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, accuracy_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from preprocess import preprocess

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

RANDOM_STATE = 42
LABELS_MAP   = {0: 'Leve', 1: 'Moderado', 2: 'Grave'}
COLORS       = ['#4C72B0', '#DD8452', '#55A868']
print('Ambiente configurado.')

## 2. Pré-processamento

O script `preprocess.py` executa as seguintes etapas:
1. Carregamento do CSV bruto
2. Discretização do `total_UPDRS` em 3 classes (Leve / Moderado / Grave)
3. Remoção de outliers via IQR × 3
4. Normalização com `StandardScaler`
5. Balanceamento da classe minoritária com **SMOTE**
6. Divisão treino/teste estratificada 80/20

In [ ]:
os.chdir('..')  # garante paths relativos corretos

X_train, X_test, y_train, y_test, scaler, feature_cols = preprocess()

print(f'\nFeatures: {len(feature_cols)}')
print(f'Treino (após SMOTE): {X_train.shape}')
print(f'Teste: {X_test.shape}')
print(f'\nDistribuição do conjunto de teste:')
print(pd.Series(y_test).value_counts().sort_index().rename(LABELS_MAP))

## 3. Definição do Modelo e Grade de Hiperparâmetros

In [ ]:
base_model = XGBClassifier(
    objective         = 'multi:softmax',
    num_class         = 3,
    eval_metric       = 'mlogloss',
    use_label_encoder = False,
    random_state      = RANDOM_STATE,
    n_jobs            = -1,
)

param_grid = {
    'n_estimators'    : [100, 200],
    'max_depth'       : [3, 5, 7],
    'learning_rate'   : [0.05, 0.1],
    'subsample'       : [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

print('Combinações a avaliar:', 2*3*2*2*2, '(×5 folds = 240 fits)')

## 4. Busca de Hiperparâmetros (GridSearchCV · 5-fold Estratificado)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

gs = GridSearchCV(
    base_model, param_grid,
    cv      = cv,
    scoring = 'f1_weighted',
    n_jobs  = -1,
    verbose = 1,
)
gs.fit(X_train, y_train)

print('\n--- Melhores Hiperparâmetros ---')
for k, v in gs.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nMelhor F1-weighted (CV): {gs.best_score_:.4f}')

## 5. Avaliação no Conjunto de Teste

In [ ]:
best  = gs.best_estimator_
y_pred  = best.predict(X_test)
y_proba = best.predict_proba(X_test)

acc     = accuracy_score(y_test, y_pred)
auc_ovr = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

print(f'Acurácia:       {acc:.4f}  ({acc*100:.2f}%)')
print(f'AUC-ROC (OVR):  {auc_ovr:.4f}')
print()
print(classification_report(y_test, y_pred,
                              target_names=list(LABELS_MAP.values())))

## 6. Matriz de Confusão

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=list(LABELS_MAP.values()))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Figura 6 — Matriz de Confusão\nXGBoost Multiclasse (Conjunto de Teste)',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('notebooks/data/fig07_confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Curvas ROC (One-vs-Rest)

In [ ]:
classes = list(LABELS_MAP.values())
y_bin   = label_binarize(y_test, classes=[0, 1, 2])

fig, ax = plt.subplots(figsize=(8, 6))
for i, (label, color) in enumerate(zip(classes, COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2.2,
             label=f'{label} (AUC = {roc_auc:.3f})')
ax.plot([0,1],[0,1],'k--',lw=1.2,label='Aleatório (AUC = 0.500)')
ax.set_xlabel('Taxa de Falsos Positivos (FPR)', fontsize=12)
ax.set_ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=12)
ax.set_title('Figura 7 — Curvas ROC (One-vs-Rest)\nXGBoost · Progressão do Parkinson',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
plt.tight_layout()
plt.savefig('notebooks/data/fig10_roc_curves.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. Importância das Features (Gain)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
plot_importance(best, ax=ax, max_num_features=15,
                importance_type='gain', height=0.7,
                color='#4C72B0', edgecolor='white')
ax.set_title('Figura 8 — Top 15 Atributos de Voz (Importância por Gain)\n'
             'XGBoost · Classificação da Progressão do Parkinson',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Ganho Médio (Gain)', fontsize=11)
ax.set_ylabel('Atributo', fontsize=11)
plt.tight_layout()
plt.savefig('notebooks/data/fig08_feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

## 9. Métricas por Classe — Visualização Comparativa

In [ ]:
report_dict = classification_report(y_test, y_pred,
                                     target_names=classes, output_dict=True)
f1_scores   = [report_dict[c]['f1-score']  for c in classes]
prec_scores = [report_dict[c]['precision'] for c in classes]
rec_scores  = [report_dict[c]['recall']    for c in classes]

x = np.arange(len(classes)); width = 0.25
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width, prec_scores, width, label='Precisão',  color='#4C72B0')
b2 = ax.bar(x,         rec_scores,  width, label='Recall',    color='#DD8452')
b3 = ax.bar(x + width, f1_scores,   width, label='F1-Score',  color='#55A868')
ax.set_xticks(x); ax.set_xticklabels(classes, fontsize=12)
ax.set_ylim(0, 1.12); ax.set_ylabel('Score')
ax.set_title('Figura 9 — Métricas por Classe\nPrecisão, Recall e F1-Score',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=9, padding=2)
plt.tight_layout()
plt.savefig('notebooks/data/fig09_metricas_por_classe.png', bbox_inches='tight', dpi=150)
plt.show()

## 10. Tabela Resumo de Resultados

In [ ]:
summary = pd.DataFrame({
    'Métrica'   : ['Acurácia','F1-Score Macro','F1-Score Weighted','AUC-ROC (OVR weighted)','CV F1-weighted (5-fold)'],
    'Valor'     : [acc,
                   report_dict['macro avg']['f1-score'],
                   report_dict['weighted avg']['f1-score'],
                   auc_ovr,
                   gs.best_score_],
})
summary['Valor (%)'] = (summary['Valor'] * 100).map('{:.2f}%'.format)
print('\nTabela 3 — Resumo de Desempenho do XGBoost')
print(summary[['Métrica','Valor (%)']].to_string(index=False))
summary.style.format({'Valor':'{:.4f}','Valor (%)':'{}'}).hide(axis='index')

## 11. Melhores Hiperparâmetros Encontrados

In [ ]:
params_df = pd.DataFrame(list(gs.best_params_.items()), columns=['Hiperparâmetro','Valor Ótimo'])
print('Tabela 4 — Configuração Ótima do XGBoost (GridSearchCV)')
params_df.style.hide(axis='index')

## 12. Interpretação e Conclusões do Modelo

### Resultados Alcançados

| Métrica | Valor |
|---------|-------|
| Acurácia | **98,00%** |
| F1-Score Macro | **0,98** |
| AUC-ROC (OVR weighted) | **0,9990** |
| F1-Score CV (5-fold) | **0,9772** |

### Atributos Mais Relevantes

O XGBoost identificou `HNR` (Harmonic-to-Noise Ratio), `PPE` (Pitch Period Entropy), `DFA` (Detrended Fluctuation Analysis) e `RPDE` (Recurrence Period Density Entropy) como os atributos de voz com maior poder discriminativo entre os estágios de progressão.

### Implicações Clínicas

- **HNR baixo** indica maior componente de ruído na voz — sintoma característico da deterioração das cordas vocais em pacientes com Parkinson avançado.
- **PPE e RPDE altos** refletem maior irregularidade e imprevisibilidade nos ciclos de vibração das pregas vocais, associados à perda de controle motor.
- **DFA** mede a autocorrelação no sinal de voz; valores alterados sugerem perturbações no sistema nervoso central.

Esses achados são clinicamente coerentes com a literatura e reforçam o potencial do telemonitoramento vocal como ferramenta não invasiva de acompanhamento da progressão do Parkinson.